# Homework: scoring clusters on your track's data

*About 2 hours. 10 points. Individual submission on Canvas, due **Monday November 9, 11:59 pm**.*

k-means always returns clusters, whether or not the data contain groups. This homework is about the three numbers that tell you what those clusters are worth: **homogeneity**, **completeness** and the **silhouette**. You compute them by hand on small tables, write them in code, and then use them on the real data of your track from [Session 7](../Chapter2-DataManipulation/lab_s07_start_here.md).

**Before you start.** [Notebook 3.3](3.3_clustering.ipynb), sections 8 to 16 (section 13 for PCA), and the [clustering slides](https://geo-smart.github.io/mlgeo-book/slides/2026/lec13_clustering.html) from Session 13 (the silhouette slide and the slide on scoring clusters against labels). All the definitions you need are repeated below.

**Learning outcomes.** After this homework you can:

1. compute homogeneity and completeness from a table of counts, and the silhouette of one data sample, by hand and in code;
2. say what information each score uses, and so what each one cannot see;
3. read the three scores on real data and explain, in physical terms, why they disagree;
4. decide which scores to report for a clustering in your own project;
5. check an AI reviewer's report against your own calculations, and say how far to trust it.

| Part | What you do | Time (min) |
|---|---|---|
| Setup | Copy the notebook, choose your track | 5 |
| 1 | Homogeneity, completeness and the silhouette by hand, then in code | 35 |
| 2 | Run your track's data cell: what one data sample is, the features, the labels | 5 |
| 3 | Scores for K = 2 to 8; one clustering read cluster by cluster | 25 |
| 4 | Change the labels, change the features, keep fewer principal components: which scores move? | 20 |
| 5 | Your own answer, an AI review of your work, your check of the reviewer, the three reflection questions | 30 |

**AI modes** ([1.8](../Chapter1-GettingStarted/1.8_ai_in_your_workflow.md)). 🔒 *By hand:* Part 1's calculations, your written answers about what your clusters mean, and your own answer and judgment in Part 5. 🤝 *Assistant allowed:* the code in Parts 1b, 1c, 3 and 4; verify every line and disclose the use. 🔍 *Agent, then verify:* the AI review in Part 5, which you then check.

**How the cells work.** In the cells marked **Your turn** you replace each `None` with a few lines of your own. The cells below them stay quiet until your answer is there. This is graded work, so the notebook has no worked solutions and no reference values; Part 1 tells you whether your numbers match scikit-learn, and in Parts 3 and 4 the check is your physical reasoning.

**Start here.** Work in a copy with your NetID in its name; git ignores it:

    cp book/Chapter3-MachineLearning/Homework_Clustering.ipynb book/Chapter3-MachineLearning/Homework_Clustering_UWNETID.ipynb

If you downloaded this notebook alone, it fetches the course data it needs from GitHub and checks the fingerprint of each file.

## Setup

Take the track of your project's domain, as in Session 7. If your domain is not one of the four, take the track whose data look most like yours.

| `TRACK` | One data sample | Features | Known labels |
|---|---|---|---|
| `"volcano"` | a located earthquake at Mount St Helens, 2004 to 2008 | depth, magnitude, waiting time since the previous earthquake | stage of the eruption |
| `"soil_moisture"` | a day at the Turnbull station, 2015 to 2024 | soil moisture at 5, 10, 20, 50, 100 cm | season |
| `"land_surface_model"` | a land grid cell of the western United States, 1991 to 2020 | mean soil water in each of the 12 calendar months | class of annual precipitation |
| `"planetary"` | a confirmed planet | mass, distance from Earth | discovery method |

In [ ]:
NETID = ""    # your UW NetID, for example "jdoe42"
TRACK = ""    # "volcano", "soil_moisture", "land_surface_model" or "planetary"

The first code cell imports every package the notebook uses, so a missing package fails here rather than halfway through.

In [ ]:
# Data handling
import hashlib
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

# Plotting
import matplotlib.pyplot as plt

# Clustering and its scores (scikit-learn)
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (homogeneity_completeness_v_measure, silhouette_samples,
                             silhouette_score)
from sklearn.preprocessing import StandardScaler

These values are used in several places. Everyone uses the same `SEED` and `N_STARTS`, so two students with the same track and the same K get the same clusters.

In [ ]:
SEED = 42
N_STARTS = 10         # random starts of k-means for each K; the run with the smallest inertia is kept
KS = range(2, 9)      # the numbers of clusters to try
TOLERANCE = 0.005     # how close two numbers must be to count as a match in Part 1

# Okabe-Ito colors, as in notebook 3.3, plus a different marker for each group so figures read in black and white
COLORS = ["#0072B2", "#E69F00", "#009E73", "#D55E00", "#CC79A7", "#56B4E9", "#6e675c", "#000000", "#b8b09f", "#8c510a"]
MARKERS = ["o", "s", "^", "D", "v", "P", "X", "*", "<", ">"]


def answered(*values):
    """True when none of these values is still None."""
    return all(value is not None for value in values)

This function finds a course data file: in the book's `snapshots/` folder if you have a clone, otherwise on GitHub. It then compares the file's SHA-256 fingerprint with the value recorded when the course copy was made, the check from Session 7.

In [ ]:
SNAPSHOTS = Path("../Chapter2-DataManipulation/snapshots")
BASE_URL = "https://raw.githubusercontent.com/geo-smart/mlgeo-book/main/book/Chapter2-DataManipulation/snapshots/"
FINGERPRINTS = {
    "comcat_msh_2004-2008.csv": "c3c99041155a7251827dad2c0a056f26a15c3bca101dfd2c9fabfc91073481d2",
    "uscrn_WA_Spokane_17_SSW_2015-2024.txt": "115f60bc36f8aed6ea8d916da42b5e14bb0a5361ecbf055b6cbbedba64dc994e",
    "uscrn_daily01_HEADERS.txt": "dde1b4d78d6b68b9a558d042876340c0476b73c962896ff1b152e8e49379946e",
    "cpc_soilw_precl_westUS_1991-2020.nc": "7c8fda8209cfb8cc5fc4c3181382c64b56c2a8ff7971302b689bc4ec8fcf9d7b",
    "cpc_westUS_static_1991-2020.nc": "a9108d71d264a3767fe6642f2af69afc332c8fe56743ab4598c216b10775671a",
    "nea_pscomppars_2026-10-14.csv": "fd1d4fa7b28deeb79e55cbe3ceeae124153b5a0be6a2fcb4679bd99b05c2401a",
    "naturalearth_pnw_map.csv": "7ed9da2e9ac642e9bc39c9d966df9b63bf169d3819ea9c588c56211b59601791",
}


def course_file(name):
    path = SNAPSHOTS / name
    if not path.exists():
        path = Path(name)
        if not path.exists():
            urllib.request.urlretrieve(BASE_URL + name, path)
    if hashlib.sha256(path.read_bytes()).hexdigest() != FINGERPRINTS[name]:
        raise ValueError(f"{path} is not the course copy. Delete it, or restore it with git, and run this cell again.")
    return path

## Part 1. The three scores by hand, then in code (about 35 min)

```{admonition} 🔒 By hand
:class: attention
Assistant closed for these numbers. Do the calculations on paper or with a calculator before you run anything, and write the arithmetic of one table in the answer cell.
```

### Homogeneity and completeness

A survey clustered 12 rock samples without looking at the lithology. Each table counts the samples by known label (rows) and cluster (columns).

| Table A | cluster 0 | cluster 1 | cluster 2 |
|---|---|---|---|
| basalt | 4 | 2 | 0 |
| granite | 0 | 0 | 6 |

| Table B | cluster 0 | cluster 1 |
|---|---|---|
| basalt | 4 | 0 |
| andesite | 2 | 0 |
| granite | 0 | 6 |

| Table C | cluster 0 | cluster 1 |
|---|---|---|
| basalt | 5 | 1 |
| granite | 1 | 5 |

**Entropy** measures how mixed a group is. If a group splits into fractions $p_1, p_2, \dots$, then

$$H = -\sum_j p_j \ln p_j .$$

It is 0 when the group is all one kind (take $0 \ln 0 = 0$) and largest when the kinds are equally common.

**Homogeneity** asks: *if I know the cluster, how much uncertainty about the label is left?*

$$h = 1 - \frac{H(\text{label} \mid \text{cluster})}{H(\text{label})},
\qquad
H(\text{label} \mid \text{cluster}) = \sum_{\text{clusters } k} \frac{n_k}{N}\, H(\text{labels inside cluster } k).$$

$H(\text{label})$ is the entropy of the row totals; $n_k$ is the number of samples in cluster $k$ and $N$ the total. $h = 1$ means every cluster is pure.

**Completeness** swaps the two roles: *if I know the label, how much uncertainty about the cluster is left?*

$$c = 1 - \frac{H(\text{cluster} \mid \text{label})}{H(\text{cluster})}.$$

$c = 1$ means no label is split across clusters. The **V-measure** is their harmonic mean, $V = 2hc/(h + c)$ (Rosenberg and Hirschberg, 2007).

**Question 1.1.** Before calculating: in which table is $h = 1$? In which is $c = 1$? In which is neither?

**Question 1.2.** Calculate $h$ and $c$ for Tables A and C, to three decimals. Write out the steps for Table A in the answer cell below.

**Question 1.3.** Table B is Table A with rows and columns swapped. Without calculating anything, give $h$ and $c$ for Table B, and explain in one sentence why you can.

*Your answers to 1.1 to 1.3, with the steps for Table A:*

**Your turn 1a.** Enter your hand values. These are the numbers from your paper, not from code.

In [ ]:
hand = {
    "A": (None, None),   # (homogeneity, completeness) of Table A
    "B": (None, None),
    "C": (None, None),
}

**Your turn 1b.** Write the same calculation as functions that take a table of counts. Rows are the labels and columns are the clusters.

*Hint:* `np.sum(p * np.log(p))`; `table.sum(axis=1)` gives the row totals and `table.sum(axis=0)` the column totals; `table[:, k]` is column `k`, which you can use in a loop or a sum over `range(table.shape[1])`. For completeness, think about what `table.T` does and what Question 1.3 told you.

In [ ]:
def entropy(counts):
    """Entropy (natural logarithm) of a group split into these counts."""
    counts = np.asarray(counts, dtype=float)
    p = counts[counts > 0] / counts.sum()
    return None        # replace None: the entropy of the fractions p


def homogeneity(table):
    table = np.asarray(table, dtype=float)
    N = table.sum()
    H_label = None                # the entropy of the row totals
    H_label_given_cluster = None  # the weighted sum over clusters
    return None        # replace None: h


def completeness(table):
    return None        # replace None: one line

This cell compares your hand values and your functions with scikit-learn. It prints `match` or `differs`, not the numbers.

In [ ]:
TABLES = {
    "A": np.array([[4, 2, 0], [0, 0, 6]]),
    "B": np.array([[4, 0], [2, 0], [0, 6]]),
    "C": np.array([[5, 1], [1, 5]]),
}


def table_to_samples(table):
    """Turn a table of counts into one label and one cluster for each sample, the form scikit-learn wants."""
    labels, clusters = np.indices(table.shape)
    return np.repeat(labels.ravel(), table.ravel()), np.repeat(clusters.ravel(), table.ravel())


def compare(mine, reference):
    if not answered(mine):
        return "not answered yet"
    return "match" if abs(mine - reference) <= TOLERANCE else "differs"


for name, table in TABLES.items():
    h, c, _ = homogeneity_completeness_v_measure(*table_to_samples(table))
    print(f"Table {name}.  By hand: h {compare(hand[name][0], h)}, c {compare(hand[name][1], c)}.  "
          f"Your functions: h {compare(homogeneity(table), h)}, c {compare(completeness(table), c)}.")

### The silhouette

Homogeneity and completeness need known labels. The **silhouette** does not: it uses only the distances between data samples and the cluster assignments (Rousseeuw, 1987). For one sample $i$:

- $a(i)$ is its mean distance to the *other* samples in its own cluster;
- $b(i)$ is its mean distance to the samples of the nearest other cluster, the cluster with the smallest such mean;

$$s(i) = \frac{b(i) - a(i)}{\max\{a(i),\, b(i)\}}, \qquad -1 \le s(i) \le 1.$$

$s$ near 1: well inside its cluster. Near 0: on the boundary between two clusters. Negative: closer to another cluster than to its own. The mean of $s$ over all samples summarizes a clustering (Kaufman and Rousseeuw, 1990):

| Mean silhouette | above 0.70 | 0.51 to 0.70 | 0.26 to 0.50 | 0.25 or less |
|---|---|---|---|---|
| Structure | strong | reasonable | weak, could be artificial | none found |

Seven earthquakes lie along a vertical line, at these depths and in these clusters, and the distance is the difference in depth:

| Cluster | Depths (km) |
|---|---|
| A | 1, 2, 3 |
| B | 5, 6 |
| C | 12, 13 |

**Question 1.4.** By hand, find $a$, $b$ and $s$ for the earthquake at 3 km, and for the one at 5 km.

**Question 1.5.** Now move the 3 km earthquake into cluster B, leaving the other assignments as they were. What is its silhouette? What does the sign tell you, and what would one step of k-means do to this earthquake?

*Your answers to 1.4 and 1.5, with $a$ and $b$ each time:*

**Your turn 1c.** Enter your hand values, then write the silhouette of one sample. `x` holds the depths and `clusters` the cluster of each.

*Hint:* `np.abs(x[mask] - x[i]).mean()` is the mean distance from sample `i` to the samples selected by a True/False `mask`. Start the mask for `a` from `clusters == clusters[i]` and set `mask[i] = False`, so the sample is not counted against itself. For `b`, take the smallest such mean over `np.unique(clusters)`, skipping the sample's own cluster.

In [ ]:
s_hand = {"3 km in A": None, "5 km in B": None, "3 km moved to B": None}


def silhouette_one(i, x, clusters):
    a = None           # replace None: mean distance to the others in its cluster
    b = None           # replace None: mean distance to the nearest other cluster
    return None        # replace None: s

In [ ]:
depth_km = np.array([1, 2, 3, 5, 6, 12, 13], dtype=float)
start = np.array([0, 0, 0, 1, 1, 2, 2])    # clusters A, B, C
moved = np.array([0, 0, 1, 1, 1, 2, 2])    # the 3 km earthquake moved to B

for name, i, assigned in [("3 km in A", 2, start), ("5 km in B", 3, start), ("3 km moved to B", 2, moved)]:
    reference = silhouette_samples(depth_km.reshape(-1, 1), assigned)[i]
    print(f"{name}.  By hand: {compare(s_hand[name], reference)}.  "
          f"Your function: {compare(silhouette_one(i, depth_km, assigned.copy()), reference)}.")

## Part 2. Your track's data (about 5 min)

Read the section of your track and run all four code cells; only yours does anything. Each one defines the same names, so the rest of the notebook is the same for everyone:

| Name | What it holds |
|---|---|
| `X` | the features, one row per data sample |
| `y` | the known labels; k-means never sees them |
| `X2` | a second choice of features (Part 4) |
| `y2` | a second set of labels (Part 4) |
| `view` | two physical coordinates for plotting the clusters |

In [ ]:
X, y, X2, y2, view = None, None, None, None, None

### Volcano: Mount St Helens, 2004 to 2008

📍 In-situ sensor · 🗂️ Catalog & compiled product · 🪨 Solid Earth · 🌋 Volcanology

On 23 September 2004 shallow earthquakes began under the crater after almost 18 years of quiet. Small explosions followed, then lava pushed up through the crater floor as solid spines until early 2008. From 16 October 2004 came the regular "drumbeat" earthquakes, which became irregular on 20 July 2005 (Moran and others, 2008; Dzurisin and others, 2015).

**One data sample** is an earthquake in the U.S. Geological Survey (USGS) Comprehensive Catalog (ComCat) box around the volcano. **Features:** depth (km), magnitude ($M_d$), and the base-10 logarithm of the waiting time since the previous located earthquake (s). **Labels `y`:** the stage of the eruption on that date (background; swarm and explosions, 23 September to 15 October 2004; regular drumbeats, to 19 July 2005; irregular drumbeats, to 31 January 2008). **`y2`:** dome building or not, the label from Part 7 of your Session 7 lab. **`X2`:** the features without depth.

Keep in mind that the Pacific Northwest Seismic Network could locate only a subset of the earthquakes during the busiest periods, so the waiting time also records how many earthquakes analysts could process.

In [ ]:
if TRACK == "volcano":
    catalog = pd.read_csv(course_file("comcat_msh_2004-2008.csv"), parse_dates=["time"]).sort_values("time")
    catalog = catalog[catalog["type"] == "earthquake"]
    catalog["time"] = catalog["time"].dt.tz_localize(None)
    catalog["log10_wait_s"] = np.log10(catalog["time"].diff().dt.total_seconds().where(lambda wait: wait > 0))
    catalog = catalog.dropna(subset=["depth", "mag", "log10_wait_s"]).reset_index(drop=True)

    X = catalog[["depth", "mag", "log10_wait_s"]]
    X2 = catalog[["mag", "log10_wait_s"]]
    stage = pd.cut(catalog["time"], pd.to_datetime(["2000-01-01", "2004-09-23", "2004-10-16", "2005-07-20", "2008-02-01", "2010-01-01"]),
                   right=False, labels=["background", "swarm", "regular drumbeats", "irregular drumbeats", "after"])
    y = stage.astype(str).replace("after", "background").to_numpy()
    y2 = np.where(np.isin(y, ["regular drumbeats", "irregular drumbeats"]), "dome building", "no dome building")
    view = {"x": catalog["time"], "y": catalog["mag"], "xlabel": "Date", "ylabel": "Magnitude $M_d$", "size": 4}

    coast = pd.read_csv(course_file("naturalearth_pnw_map.csv"))
    fig, ax = plt.subplots(figsize=(6, 4.5))
    for _, part in coast[coast["part"] >= 0].groupby(["kind", "part"]):
        ax.plot(part["lon"], part["lat"], color="#6e675c", lw=0.7)
    ax.plot(catalog["longitude"], catalog["latitude"], "o", color=COLORS[3], ms=3, label="earthquakes in the catalog")
    ax.set(xlabel="Longitude", ylabel="Latitude", title="Mount St Helens in the Pacific Northwest (ComCat)", aspect=1.45)
    ax.legend(loc="upper right")
    plt.show()

### Soil moisture: water in the top metre of soil at Turnbull

📍 In-situ sensor · 💧 Land water · 💧 Hydrology & land-surface modeling

The top metre of soil is a small reservoir. Rain and snowmelt fill it; evaporation, root uptake and drainage empty it. The U.S. Climate Reference Network (USCRN) station WA Spokane 17 SSW sits in Turnbull National Wildlife Refuge, where summers are dry and winters wet and cold (Bell and others, 2013).

**One data sample** is a day, 2015 to 2024. **Features:** the daily soil moisture at 5, 10, 20, 50 and 100 cm (m³/m³). **Labels `y`:** the meteorological season (DJF, MAM, JJA, SON). **`y2`:** the calendar year, the label for "did one year differ from another?". **`X2`:** the five depths plus the daily mean air temperature.

Keep in mind that a day enters only if all five depths and the temperature report a value. Most of the dropped days are in winter, as you saw with the frozen days in Session 7.

In [ ]:
if TRACK == "soil_moisture":
    columns = course_file("uscrn_daily01_HEADERS.txt").read_text().splitlines()[1].split()
    daily = pd.read_csv(course_file("uscrn_WA_Spokane_17_SSW_2015-2024.txt"), sep=r"\s+", names=columns,
                        dtype={"LST_DATE": str})
    daily["date"] = pd.to_datetime(daily["LST_DATE"])
    depths = ["SOIL_MOISTURE_5_DAILY", "SOIL_MOISTURE_10_DAILY", "SOIL_MOISTURE_20_DAILY",
              "SOIL_MOISTURE_50_DAILY", "SOIL_MOISTURE_100_DAILY"]
    daily = daily.replace([-99, -9999], np.nan).dropna(subset=depths + ["T_DAILY_MEAN"]).reset_index(drop=True)

    X = daily[depths]
    X2 = daily[depths + ["T_DAILY_MEAN"]]
    seasons = {12: "DJF", 1: "DJF", 2: "DJF", 3: "MAM", 4: "MAM", 5: "MAM", 6: "JJA", 7: "JJA", 8: "JJA",
               9: "SON", 10: "SON", 11: "SON"}
    y = daily["date"].dt.month.map(seasons).to_numpy()
    y2 = daily["date"].dt.year.astype(str).to_numpy()
    view = {"x": daily["date"].dt.dayofyear, "y": daily["SOIL_MOISTURE_5_DAILY"], "xlabel": "Day of the year",
            "ylabel": "Soil moisture at 5 cm (m³/m³)", "size": 8}

    coast = pd.read_csv(course_file("naturalearth_pnw_map.csv"))
    fig, ax = plt.subplots(figsize=(6, 4.5))
    for _, part in coast[coast["part"] >= 0].groupby(["kind", "part"]):
        ax.plot(part["lon"], part["lat"], color="#6e675c", lw=0.7)
    ax.plot(daily["LONGITUDE"].iloc[0], daily["LATITUDE"].iloc[0], "*", color=COLORS[3], ms=14, label="WA Spokane 17 SSW (USCRN)")
    ax.set(xlabel="Longitude", ylabel="Latitude", title="The Turnbull station", aspect=1.45)
    ax.legend(loc="upper right")
    plt.show()

### Land-surface model: soil water in a model of the western United States

🖥️ Model & simulation · 💧 Land water · 💧 Hydrology & land-surface modeling

No instrument measures the whole root zone across the West, so NOAA's Climate Prediction Center (CPC) computes soil water with a one-layer model driven by observed precipitation and temperature (Fan and van den Dool, 2004).

**One data sample** is a land grid cell (0.5°, more than half land). **Features:** the 1991 to 2020 mean soil water (mm) of each of the 12 calendar months, so a sample is a mean seasonal cycle. **Labels `y`:** the tercile of the cell's annual precipitation (dry, middle, wet). **`y2`:** the cell's wettest season (DJF, MAM, JJA, SON). **`X2`:** the same 12 months after removing each cell's own mean and dividing by its own standard deviation, which keeps the timing of the seasonal cycle and removes its size.

Keep in mind that the labels come from the precipitation that drives the model, so this is a test of your Session 7 question: how much of the soil water does precipitation alone explain?

In [ ]:
if TRACK == "land_surface_model":
    grid = xr.open_dataset(course_file("cpc_soilw_precl_westUS_1991-2020.nc"))
    static = xr.open_dataset(course_file("cpc_westUS_static_1991-2020.nc"))
    soil_water = grid["soilw"].groupby("time.month").mean()
    precip = grid["precip"].groupby("time.month").mean()
    land = (static["landperc"] > 50) & soil_water.notnull().all("month")

    cells = land.to_dataframe("land").reset_index().query("land")
    pick = {"lat": xr.DataArray(cells["lat"].to_numpy(), dims="cell"), "lon": xr.DataArray(cells["lon"].to_numpy(), dims="cell")}
    cycle = soil_water.sel(pick).transpose("cell", "month").to_numpy()
    rain = precip.sel(pick).transpose("cell", "month").to_numpy()

    months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
    X = pd.DataFrame(cycle, columns=months)
    X2 = pd.DataFrame((cycle - cycle.mean(axis=1, keepdims=True)) / cycle.std(axis=1, keepdims=True), columns=months)
    y = pd.qcut(rain.mean(axis=1), 3, labels=["dry", "middle", "wet"]).astype(str)
    y = np.asarray(y)
    season_rain = pd.DataFrame({"DJF": rain[:, [11, 0, 1]].sum(axis=1), "MAM": rain[:, [2, 3, 4]].sum(axis=1),
                                "JJA": rain[:, [5, 6, 7]].sum(axis=1), "SON": rain[:, [8, 9, 10]].sum(axis=1)})
    y2 = season_rain.idxmax(axis=1).to_numpy()
    view = {"x": cells["lon"].to_numpy() - 360, "y": cells["lat"].to_numpy(), "xlabel": "Longitude", "ylabel": "Latitude",
            "size": 22, "marker": "s"}

    fig, ax = plt.subplots(figsize=(5.5, 5))
    mesh = ax.pcolormesh(soil_water["lon"] - 360, soil_water["lat"], soil_water.mean("month").where(land), cmap="Blues", shading="nearest")
    fig.colorbar(mesh, ax=ax, label="Mean soil water, 1991 to 2020 (mm)")
    ax.set(xlabel="Longitude", ylabel="Latitude", title="The CPC model grid", aspect=1.3)
    plt.show()

### Planetary science: planets around other stars

🔭 Telescope · 🗂️ Catalog & compiled product · 🪐 Planets & exoplanets · 🪐 Planetary science & astrobiology

The NASA Exoplanet Archive lists the confirmed planets, and each method of finding planets favors its own kind: transits favor planets close to their star, radial velocity favors massive planets around nearby stars, imaging favors young giants far from their star, and microlensing reaches distant stars. Your Session 7 question was which patterns come from the planets and which from what telescopes can detect.

**One data sample** is a confirmed planet found by one of the four main methods. **Features:** the base-10 logarithms of the planet's mass (Earth masses) and of its distance from Earth (parsecs). The mass column holds the best estimate available, which may be a minimum mass $M \sin i$. **Labels `y`:** the discovery method. **`y2`:** a mass class (below 10, 10 to 100, above 100 Earth masses). **`X2`:** mass, distance and radius.

Keep in mind that most radii of planets not found by transit have no uncertainty in the table (look at `pl_radeerr1`), which suggests they were estimated from the mass rather than measured.

In [ ]:
if TRACK == "planetary":
    planets = pd.read_csv(course_file("nea_pscomppars_2026-10-14.csv"), comment="#")
    planets = planets[planets["discoverymethod"].isin(["Transit", "Radial Velocity", "Microlensing", "Imaging"])]
    planets = planets.dropna(subset=["pl_bmasse", "sy_dist", "pl_rade"]).reset_index(drop=True)

    logs = pd.DataFrame({"log10_mass": np.log10(planets["pl_bmasse"]), "log10_distance": np.log10(planets["sy_dist"]),
                         "log10_radius": np.log10(planets["pl_rade"])})
    X = logs[["log10_mass", "log10_distance"]]
    X2 = logs
    y = planets["discoverymethod"].to_numpy()
    y2 = np.select([planets["pl_bmasse"] < 10, planets["pl_bmasse"] < 100], ["below 10", "10 to 100"], "above 100")
    view = {"x": logs["log10_distance"], "y": logs["log10_mass"], "xlabel": "log10 distance from Earth (pc)",
            "ylabel": "log10 mass (Earth masses)", "size": 6}

    fig, ax = plt.subplots(figsize=(7, 3.8))
    for j, method in enumerate(["Transit", "Radial Velocity", "Microlensing", "Imaging"]):
        chosen = planets["discoverymethod"] == method
        ax.scatter(planets.loc[chosen, "ra"], planets.loc[chosen, "dec"], s=6, color=COLORS[j], marker=MARKERS[j], label=method)
    ax.set(xlabel="Right ascension (degrees)", ylabel="Declination (degrees)", title="Where the host stars are on the sky", xlim=(360, 0))
    ax.legend(loc="lower right", markerscale=2)
    plt.show()

In [ ]:
if answered(X, y):
    print(f"Track {TRACK}: {len(X)} data samples, {X.shape[1]} features.\n")
    print(X.describe().round(3).loc[["min", "50%", "max"]].to_string())
    print("\nHow many data samples carry each label:")
    print(pd.Series(y).value_counts().to_string())
else:
    print('Set TRACK in the Setup section and run the cells again from there.')

## Part 3. What the scores say on your data (about 25 min)

Before you run anything, write one sentence: which K do you expect, and which labels do you expect the clusters to recover or to mix?

*Your prediction:*

**Your turn 3a.** Standardize the features, as k-means needs (3.3), and write a function that clusters for each K and scores the result.

*Hint:* `StandardScaler().fit_transform(X)`. In the loop, `KMeans(n_clusters=k, n_init=N_STARTS, random_state=SEED).fit_predict(Z)`, then `silhouette_score(Z, clusters)` and `homogeneity_completeness_v_measure(y, clusters)`, which returns three numbers with the known labels first. Section 16 of notebook 3.3 has the same loop.

In [ ]:
Z = None     # replace None: the standardized features


def scores_by_k(Z, y, ks):
    rows = []
    for k in ks:
        clusters = None         # replace None: the k-means clusters for this k
        sil = None              # replace None: the mean silhouette
        h, c, v = None, None, None   # replace: homogeneity, completeness, V-measure
        rows.append({"K": k, "silhouette": sil, "homogeneity": h, "completeness": c, "v_measure": v})
    return pd.DataFrame(rows).set_index("K")

This cell runs your function and draws the scores against K. It takes a few seconds on most tracks and up to half a minute on the volcano track.

In [ ]:
def plot_scores(scores):
    fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4), sharex=True)
    left.plot(scores.index, scores["silhouette"], marker="o", color=COLORS[0], lw=2)
    left.set(xlabel="Number of clusters K", ylabel="Mean silhouette", title="Uses distances only", ylim=(0, 1))
    for j, name in enumerate(["homogeneity", "completeness", "v_measure"]):
        right.plot(scores.index, scores[name], marker=MARKERS[j + 1], color=COLORS[j + 1], lw=2, label=name)
    right.set(xlabel="Number of clusters K", ylabel="Score against the labels", title="Uses the known labels", ylim=(0, 1))
    right.legend()
    plt.show()


scores = scores_by_k(Z, y, KS) if answered(Z, y) else None
if scores is None or scores.isna().any().any():
    print("Not answered yet: finish Your turn 3a.")
else:
    scores = scores.astype(float)
    print(scores.round(3).to_string())
    plot_scores(scores)

**Your turn 3b.** Read the K that each score favors from the table. Then choose the K you would defend and look at that clustering closely.

*Hint:* `scores["silhouette"].idxmax()` gives the K of the largest value. `silhouette_samples(Z, clusters)` gives one silhouette per data sample. `pd.crosstab(y, clusters)` counts the samples by label and cluster, like the tables of Part 1. `pd.Series(s).groupby(clusters).mean()` gives a mean per cluster; the mean of a True/False array is a fraction.

In [ ]:
k_silhouette = None     # replace None: the K with the largest mean silhouette
k_v_measure = None      # replace None: the K with the largest V-measure
K = None                # replace None: the K you choose, an integer

clusters = None         # replace None: the k-means clusters at K, with n_init=N_STARTS and random_state=SEED
s = None                # replace None: the silhouette of each data sample
table = None            # replace None: the crosstab of y against clusters
s_by_cluster = None     # replace None: the mean silhouette of each cluster
fraction_negative = None     # replace None: the fraction of samples with s below 0

Your Part 1 functions should give scikit-learn's numbers on this real table too. The next two figures are the silhouette diagram and your clusters in physical coordinates beside the known labels. Colors in the two panels of the second figure are unrelated: cluster numbers are arbitrary.

In [ ]:
def plot_silhouette_diagram(s, clusters):
    fig, ax = plt.subplots(figsize=(7, 4.5))
    bottom = 0
    for j in np.unique(clusters):
        values = np.sort(s[clusters == j])
        ax.fill_betweenx(np.arange(bottom, bottom + len(values)), 0, values, color=COLORS[j], lw=0)
        ax.text(-0.04, bottom + len(values) / 2, f"cluster {j}", ha="right", va="center")
        bottom += len(values) + len(s) // 50
    ax.axvline(s.mean(), color="#26241f", ls="--", lw=1.5, label=f"mean silhouette {s.mean():.2f}")
    ax.axvline(0, color="#6e675c", lw=1)
    ax.set(xlabel="Silhouette of each data sample", yticks=[], xlim=(min(-0.35, s.min()), 1), title="Silhouette diagram")
    ax.legend(loc="lower right")
    plt.show()


def plot_view(view, groups, title, ax):
    for j, group in enumerate(pd.unique(pd.Series(groups).sort_values())):
        chosen = np.asarray(groups) == group
        ax.scatter(np.asarray(view["x"])[chosen], np.asarray(view["y"])[chosen], s=view["size"], color=COLORS[j],
                   marker=view.get("marker", MARKERS[j]), label=str(group), lw=0)
    ax.set(xlabel=view["xlabel"], ylabel=view["ylabel"], title=title)
    ax.legend(markerscale=2, fontsize=9)


if answered(K, clusters, s, table):
    h, c, v = homogeneity_completeness_v_measure(y, clusters)
    print(f"K = {K}: homogeneity {h:.3f}, completeness {c:.3f}, V-measure {v:.3f}, mean silhouette {s.mean():.3f}")
    print(f"Your functions on this table: h {compare(homogeneity(table.to_numpy()), h)}, c {compare(completeness(table.to_numpy()), c)}\n")
    print(table.to_string())
    if answered(s_by_cluster, fraction_negative):
        print(f"\nMean silhouette of each cluster: {s_by_cluster.round(3).to_dict()}")
        print(f"Fraction of samples with a negative silhouette: {fraction_negative:.3f}")
    plot_silhouette_diagram(s, clusters)
    fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4.8), sharex=True, sharey=True)
    plot_view(view, clusters, f"k-means clusters, K = {K}", left)
    plot_view(view, y, "Known labels", right)
    plt.show()
else:
    print("Not answered yet: finish Your turn 3b.")

**Question 3.1.** Which K does the silhouette favor, and which K does the V-measure favor? If they differ, explain why, with the numbers. If they agree, explain why that need not happen. Use the threshold table in Part 1 to say how much structure the silhouette finds.

**Question 3.2.** Go through your clusters one by one. Using the crosstab and the physical figure, say what each cluster is in the terms of your system (for example, "the wet days of winter and spring"). Which label is split across clusters, and how does that lower completeness? Which cluster mixes labels, and how does that lower homogeneity?

**Question 3.3.** Which cluster has the lowest mean silhouette, and what fraction of samples has a negative silhouette? Where do those samples sit in physical coordinates, and what might make them hard to place?

*Your answers to 3.1, 3.2 and 3.3:*

## Part 4. Change the labels, the features, the dimension (about 20 min)

Before you run this part, predict from the definitions: if you change only the labels, which of the scores can change? If you change only the features, which can change?

*Your prediction:*

**Your turn 4.** At your K, fill a table with the scores for all four combinations of features (`X`, `X2`) and labels (`y`, `y2`).

*Hint:* a loop over the features inside which you standardize, then a loop over the labels, calling your `scores_by_k` with `[K]` as the list of K values. `.iloc[0].to_dict()` turns its one row into a dictionary. Build `combinations` as a DataFrame with one row per combination.

In [ ]:
features = {"X": X, "X2": X2}
labels = {"y": y, "y2": y2}

combinations = None     # replace None: columns features, labels, silhouette, homogeneity, completeness, v_measure

In [ ]:
if answered(combinations):
    print(combinations.round(3).to_string(index=False))
else:
    print("Not answered yet: finish Your turn 4.")

**Question 4.1.** Which scores stayed the same when only the labels changed, and which stayed the same when only the features changed? Explain each from the formulas in Part 1.

**Question 4.2.** Answer for your track.

- *Volcano:* The labels are dates. Why should earthquake depth, size and waiting time change at those dates, and which feature could reflect the catalog and not the volcano? Is a low homogeneity here a failure of k-means, of the features, or of the labels?
- *Soil moisture:* What do the scores against the calendar year (`y2`) tell you about what these clusters capture? What happens to the clusters and the scores when you add air temperature, and what does that say about choosing features?
- *Land-surface model:* Which features go with which labels, the size of the seasonal cycle with annual precipitation or its timing with the wettest season? What does that say about your Session 7 question, how much of the soil water precipitation alone explains?
- *Planetary:* The mass classes `y2` are computed from one of the features. What does that do to the scores, and how would you guard against it in your own work? Do the clusters against the discovery method (`y`) reflect the planets, the telescopes, or both? What do the radii in `X2` add, given how many of them were not measured?

*Your answers to 4.1 and 4.2:*

### Fewer dimensions

Notebook 3.3 reduced the dimension with principal component analysis (PCA) before clustering. Does that change what the three scores tell you?

**Your turn 4b.** Fit PCA on your standardized features `Z`. Then cluster at your K on the first principal component alone and on the first two, and score each against `y`.

*Hint:* `PCA().fit(Z)`; `.explained_variance_ratio_.cumsum()` gives the share of the variance kept by the first 1, 2, 3... components; `pca.transform(Z)[:, :n]` keeps the first `n` components. Your `scores_by_k` works on these arrays, with `[K]` as the list of K values.

In [ ]:
pca = None              # replace None: PCA fitted on Z
variance_kept = None    # replace None: the cumulative share of variance kept

pca_scores = None       # replace None: a DataFrame with columns components, silhouette, homogeneity, completeness, v_measure

In [ ]:
if answered(pca, variance_kept, pca_scores):
    print("Cumulative share of the variance kept:", np.round(variance_kept, 3)[:6])
    print("\nWeights of the first principal component:")
    print(pd.Series(pca.components_[0], index=X.columns).round(3).to_string())
    print(f"\nScores at K = {K}, against y:")
    print(pca_scores.round(3).to_string(index=False))
else:
    print("Not answered yet: finish Your turn 4b.")

**Question 4.3.** How much of the variance does the first component keep, and what does it represent in your system? Read its weights: what would a large value of this component mean physically? Compare the silhouette and the V-measure with your result on all the features (the row `X`, `y` above). Which moved, and which did not? Does a higher silhouette after PCA mean a better clustering? Explain from the definition of the silhouette. If your track has only two or three features, say also what PCA can and cannot do for it.

*Your answer to 4.3:*

## Part 5. Your own answer, then an AI review (about 30 min)

Order matters here. You write what you learned first, in your own words. Then an AI tool reviews your work against a rubric, as in the [AI-ready dataset assignment](../Chapter2-DataManipulation/2.20_Final_Project_Assignement.md). Then you check the reviewer, because its report is a claim like any other.

### 5a. Your own answer (about 10 min)

```{admonition} 🔒 By hand
:class: attention
Assistant closed for this step. Write these answers before you open any review, and do not edit them afterwards except as described in 5c.
```

**Question 5.1. What did you learn?** Explain to a classmate on another track, in five or six sentences of your own: what homogeneity, completeness and the silhouette each measure, and one thing your data showed that you did not expect, with its number.

**Question 5.2. Your project.** If you clustered your project's data, what would be one data sample, and which labels, even partial ones, could you score against? Which of the three scores would you report, and what would a high value not prove?

**Question 5.3. Before the review.** Which of your answers are you least sure of? What do you expect the reviewer to flag, and what do you expect it to miss?

*Your answers to 5.1, 5.2 and 5.3:*

### 5b. The AI review (about 10 min)

```{admonition} 🔍 Agent, then verify
:class: important
An agent reviews your work. You accept a finding only after you have checked it yourself, and you submit the exchange with your homework ([1.8](../Chapter1-GettingStarted/1.8_ai_in_your_workflow.md)).
```

**Step 1. Make a review copy.** Restart the kernel, run all cells and save. Then, from the `mlgeo-book` folder:

    pixi run jupyter nbconvert --to markdown book/Chapter3-MachineLearning/Homework_Clustering_UWNETID.ipynb --output review_copy --TagRemovePreprocessor.enabled=True --TagRemovePreprocessor.remove_cell_tags=not-for-review

This writes `review_copy.md` beside your notebook, with your answers, your code and the printed numbers, and without this cell of instructions. It holds the file names of your figures but not the pictures, so the reviewer judges your description of them. Git ignores it.

**Step 2. Plant one error in the review copy, not in your notebook.** Change one thing a careful reader could catch: a number in one of your written answers, or swap "homogeneity" and "completeness" in one sentence. Write down what you changed, for yourself. Do not tell the reviewer. This is the check from Session 7, a planted problem with a known answer, applied to the reviewer. Copy the reviewer's instructions below first, because the review copy leaves them out.

**Step 3. Run the review in a new conversation.** Use UW Purple, or Claude or GitHub Copilot in VS Code. Give the tool `review_copy.md` and the instructions below. Purple cannot run code, so upload the file as text and note in 5c that the tool could only read. Save the tool's whole report as `ai_review_UWNETID.md`.

```text
You are reviewing a student's homework on scoring clusters (ESS 469/569, University of Washington).
The attached file holds the student's answers, code and printed results.

How to score
- Score each criterion below 0, 1 or 2:
  0 = missing or wrong; 1 = present but incomplete; 2 = correct, and another reader could verify it
  from the evidence you quote.
- For every score, quote the evidence: the question number and the student's words, or the printed number.
- If you cannot find evidence, write "not found" and score 0. Do not infer a score from how confident
  the writing sounds.
- Recompute what you can from the definitions in the file, and say which numbers you recomputed and which
  you only read. If you cannot run code, say so.
- Do not write corrected answers. Say what is wrong or missing and where.
- Report a table with columns: criterion, score, evidence, what is wrong or missing.

Criteria
- R1 Hand calculations. Recompute Table A and the three silhouettes from the tables and the formulas in
  the file. Are the student's values and steps right?
- R2 Code. Do the functions carry out the definitions, without calling scikit-learn to get the answer?
- R3 Numbers. Does every number in the written answers match a printed result in the file?
  List any that do not.
- R4 Clusters. In 3.2 and 3.3, is each cluster named in terms of the physical system, and is the claim
  supported by the crosstab and the figures described?
- R5 Definitions. In 4.1, does the explanation follow from the formulas in Part 1? Is homogeneity anywhere
  confused with completeness?
- R6 Reasoning. In 4.2 and 4.3, does the student reason about the physics or the instruments, and avoid
  claiming more than the scores can show?
- R7 Own words. Is 5.1 specific to this student's data, with a number, or could it describe any dataset?

Before you report
- Re-read each score of 2 and confirm that the evidence you quote would let another reader check it.
- List the two criteria with the lowest scores.
```

### 5c. Check the reviewer (about 10 min)

```{admonition} 🔒 By hand
:class: attention
Assistant closed for this step: the judgment of whether the reviewer is right is yours.
```

Check at least three of the reviewer's scores or findings against your own work: rerun a cell, redo the formula, look at the crosstab. Look for a score that is wrong or not supported by the evidence it quotes, and say how you found out. If you find none, check the scores of 2 again, and report what you checked.

If a finding is right and you change an answer because of it, keep your original sentence and add below it: *Revised after AI review:* what changed and why. An honest record of a corrected error costs no points.

**Question 5.4. How did the reviewer check your work?** Which tool and model did you use? Did it recompute numbers or only read them? Did it find the error you planted? What did it say with confidence and no evidence? Would you let it grade this homework alone, and why or why not?

*Your check of the reviewer:*

| The reviewer's score or finding | How I checked it | Right, wrong or unsupported | What I changed, if anything |
|---|---|---|---|
| | | | |
| | | | |
| | | | |

*The error I planted, and whether the reviewer found it:*

*Your answer to 5.4:*

### Reflect on this assignment

As for every assignment in this course ([1.8](../Chapter1-GettingStarted/1.8_ai_in_your_workflow.md)), a few sentences each:

1. **What did you learn from this exercise?** Name the scientific or technical idea, with one example from your own work. Your 5.1 can be the start; say what changed after the review, if anything.
2. **What did you learn from interacting with agents or chatbots?** Say what you asked, what the tool got right, what it got wrong, and how you found out.
3. **What would you suggest to improve this assignment so it meets its learning goals?** Be specific.

**AI-use disclosure.** One or two sentences: which tools you used, for what, and what you verified.

*Your three answers and your disclosure:*

The last cell collects your main results in one block for grading.

In [ ]:
print("HOMEWORK CLUSTERING SUMMARY")
print(f"NetID: {NETID or 'not set'}   Track: {TRACK or 'not set'}")
if answered(K, clusters, s) and scores is not None:
    h, c, v = homogeneity_completeness_v_measure(y, clusters)
    print(f"Samples: {len(y)}   K favored by the silhouette: {k_silhouette}   by the V-measure: {k_v_measure}   chosen: {K}")
    print(f"At K = {K}: silhouette {s.mean():.3f}, homogeneity {h:.3f}, completeness {c:.3f}, V-measure {v:.3f}, "
          f"negative silhouettes {fraction_negative:.3f}")
if answered(combinations):
    print(combinations.round(3).to_string(index=False))
if answered(pca_scores):
    print(pca_scores.round(3).to_string(index=False))

## Submit by Monday November 9, 11:59 pm

Upload to the Canvas assignment **"Homework: scoring clusters"**. Each person submits two files: your notebook, `Homework_Clustering_UWNETID.ipynb`, without the planted error, and the reviewer's report, `ai_review_UWNETID.md` (a PDF of the conversation is also accepted). Before you upload the notebook, choose Kernel, Restart Kernel and Run All Cells, then save, and check that the last cell prints your NetID and track.

| Points | What is graded |
|---|---|
| 3 | **Part 1.** Hand values match, with the steps shown for Table A and $a$ and $b$ shown for the silhouettes; the functions reproduce scikit-learn; 1.3 and 1.5 are explained. |
| 2 | **Part 3.** The notebook runs; 3.1 to 3.3 use your own numbers and name what the clusters are in your system. |
| 2 | **Part 4.** 4.1 and 4.3 are explained from the definitions; 4.2 and 4.3 reason about your track's physics or instruments. |
| 1 | **Part 5a.** 5.1 is in your own words with a number from your data; 5.2 is specific to your project. |
| 1 | **Part 5b and 5c.** The report is attached; your checks are real (a rerun, a formula, a table) and report any wrong or unsupported claim, or the second pass that found none; 5.4 says how the reviewer worked. |
| 1 | **Reflection and disclosure.** Specific answers to the three questions, scored for specificity, not for agreement with the instructor. |

There is no single right K. You are graded on reasoning that follows from your numbers.

## Sources

- Rosenberg, A., and J. Hirschberg (2007), V-measure: A conditional entropy-based external cluster evaluation measure, *Proceedings of EMNLP-CoNLL 2007*, 410-420, <https://aclanthology.org/D07-1043/>.
- Rousseeuw, P. J. (1987), Silhouettes: A graphical aid to the interpretation and validation of cluster analysis, *Journal of Computational and Applied Mathematics*, 20, 53-65, [doi:10.1016/0377-0427(87)90125-7](https://doi.org/10.1016/0377-0427(87)90125-7).
- Kaufman, L., and P. J. Rousseeuw (1990), *Finding Groups in Data: An Introduction to Cluster Analysis*, Wiley.
- Volcano: Dzurisin and others (2015), *Bulletin of Volcanology* 77, 89, [doi:10.1007/s00445-015-0973-4](https://doi.org/10.1007/s00445-015-0973-4); Moran and others (2008), USGS Professional Paper 1750, chap. 2, [doi:10.3133/pp17502](https://doi.org/10.3133/pp17502). Catalog: USGS ComCat.
- Soil moisture: Palecki and others (2015), USCRN processed data, NOAA NCEI, doi:10.7289/V5MS3QR9; Bell and others (2013), *Journal of Hydrometeorology*, 14, 977-988, [doi:10.1175/JHM-D-12-0146.1](https://doi.org/10.1175/JHM-D-12-0146.1). Map: Natural Earth.
- Land-surface model: Fan, Y., and H. van den Dool (2004), *Journal of Geophysical Research*, 109, D10102, [doi:10.1029/2003JD004345](https://doi.org/10.1029/2003JD004345); data from NOAA Physical Sciences Laboratory.
- Planetary: NASA Exoplanet Archive, Planetary Systems Composite Parameters table, retrieved 14 October 2026.